# 22. FreFormer: Frequency-Domain Multivariate Time Series Forecasting

Implements **FreFormer** from:
**"FreFormer: A Multivariate Time Series Forecasting Method Based on Frequency-Domain Features"** (CBD 2025).

## Paper (adapted to 72h→24h)
- **Embedding:** Treat input as tokens (here univariate: 1 token of length 72), linear project to hidden dimension D.
- **Encoder (L layers):**
  - **Frequency Feature Extraction (FFE):** FFT → complex MLP on real/imag parts → iFFT to capture intra-series frequency features.
  - **Frequency Attention (FA):** FFT on Q/K/V along variable dimension, multi-head attention in frequency domain, iFFT (reduces complexity; for M=1 we have one frequency bin).
  - **Add & Norm & FeedForward** with residuals.
- **Projection:** Linear map to prediction length 24; inverse normalization for output.

## Same setup as 10–21
Data: work_dir/final, 72h→24h. Metrics: MAE, RMSE, MASE. Naive baseline. Same visuals.


**GPU:** Use kernel **Python 3.11 (cablelabs-3 .venv)** (Kernel → Change kernel) so TensorFlow uses the project's venv with tensorflow-metal.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'): _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU found. Change kernel to "Python 3.11 (cablelabs-3 .venv)" (Kernel → Change kernel), then re-run from the top.')
USE_GPU = len(gpus) > 0


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists(): return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try: dfs.append(pd.read_parquet(p))
        except Exception as e: print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)


In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band: continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## FreFormer: FFE (Frequency Feature Extraction) + FA (Frequency Attention) + Encoder

In [ ]:
class ComplexLinear(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        in_dim = input_shape[-1]
        self.Wr = self.add_weight('Wr', (in_dim, self.units), initializer='glorot_uniform', trainable=True)
        self.Wi = self.add_weight('Wi', (in_dim, self.units), initializer='glorot_uniform', trainable=True)
        self.Br = self.add_weight('Br', (self.units,), initializer='zeros', trainable=True)
        self.Bi = self.add_weight('Bi', (self.units,), initializer='zeros', trainable=True)
        super().build(input_shape)
    def call(self, x_complex):
        r, i = tf.math.real(x_complex), tf.math.imag(x_complex)
        out_r = tf.matmul(r, self.Wr) - tf.matmul(i, self.Wi) + self.Br
        out_i = tf.matmul(r, self.Wi) + tf.matmul(i, self.Wr) + self.Bi
        return tf.complex(out_r, out_i)

class FreFormerFFE(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.n_freq = d_model // 2 + 1
    def build(self, input_shape):
        self.complex_linear = ComplexLinear(self.n_freq)
        super().build(input_shape)
    def call(self, x):
        x_fft = tf.signal.rfft(x)
        x_mlp = self.complex_linear(x_fft)
        out = tf.signal.irfft(x_mlp)
        return out[:, :, :self.d_model]

def frequency_attention(Q, K, V, dk, m_dim):
    # Q,K,V: (batch, M, dk). FFT along M dimension -> (batch, M//2+1, dk)
    Q_t = tf.transpose(Q, [0, 2, 1])
    K_t = tf.transpose(K, [0, 2, 1])
    V_t = tf.transpose(V, [0, 2, 1])
    Qf = tf.signal.rfft(tf.cast(Q_t, tf.complex64))
    Kf = tf.signal.rfft(tf.cast(K_t, tf.complex64))
    Vf = tf.signal.rfft(tf.cast(V_t, tf.complex64))
    Qf = tf.transpose(Qf, [0, 2, 1])
    Kf = tf.transpose(Kf, [0, 2, 1])
    Vf = tf.transpose(Vf, [0, 2, 1])
    scale = tf.cast(tf.sqrt(tf.cast(dk, tf.float32)), tf.complex64)
    scores = tf.matmul(Qf, Kf, adjoint_b=True) / scale
    scores = tf.cast(tf.nn.softmax(tf.math.real(scores), axis=-1), tf.complex64)
    out_f = tf.matmul(scores, Vf)
    out_f = tf.transpose(out_f, [0, 2, 1])
    out = tf.signal.irfft(out_f, fft_length=m_dim)
    out = tf.transpose(out, [0, 2, 1])
    return out

class FreFormerFALayer(layers.Layer):
    def __init__(self, d_model, dk, num_heads=1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.dk = dk
        self.num_heads = num_heads
    def build(self, input_shape):
        self.Wq = layers.Dense(self.dk * self.num_heads, use_bias=False)
        self.Wk = layers.Dense(self.dk * self.num_heads, use_bias=False)
        self.Wv = layers.Dense(self.dk * self.num_heads, use_bias=False)
        self.Wo = layers.Dense(self.d_model, use_bias=False)
        super().build(input_shape)
    def call(self, x):
        batch, M, D = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2]
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        if self.num_heads > 1:
            Q = tf.reshape(Q, [-1, M, self.num_heads, self.dk])
            K = tf.reshape(K, [-1, M, self.num_heads, self.dk])
            V = tf.reshape(V, [-1, M, self.num_heads, self.dk])
            Q = tf.transpose(Q, [0, 2, 1, 3])
            K = tf.transpose(K, [0, 2, 1, 3])
            V = tf.transpose(V, [0, 2, 1, 3])
            E_list = []
            for h in range(self.num_heads):
                E_list.append(frequency_attention(Q[:, h], K[:, h], V[:, h], self.dk, M))
            E = tf.stack(E_list, axis=1)
            E = tf.reshape(E, [-1, M, self.dk * self.num_heads])
        else:
            E = frequency_attention(Q, K, V, self.dk, M)
        return self.Wo(E)

class FreFormerEncoderLayer(layers.Layer):
    def __init__(self, d_model, dk, num_heads, ffn_dim, **kwargs):
        super().__init__(**kwargs)
        self.ffe = FreFormerFFE(d_model)
        self.fa = FreFormerFALayer(d_model, dk, num_heads)
        self.norm1 = layers.LayerNormalization(epsilon=1e-5)
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)
        self.ffn = keras.Sequential([
            layers.Dense(ffn_dim, activation='relu'),
            layers.Dense(d_model)
        ])
    def call(self, x):
        x = x + self.ffe(x)
        s = self.norm1(x + self.fa(x))
        return self.norm2(s + self.ffn(s))


In [ ]:
D_MODEL = 64
DK = 32
NUM_HEADS = 2
FFN_DIM = 128
NUM_LAYERS = 2

def build_freformer():
    inp = layers.Input(shape=(LOOKBACK, 1))
    # Embedding: (batch, 72, 1) -> treat as (batch, M=1, L=72), project to D
    x = tf.transpose(inp, [0, 2, 1])
    x = layers.Dense(D_MODEL, name='embed')(x)
    for i in range(NUM_LAYERS):
        enc = FreFormerEncoderLayer(D_MODEL, DK, NUM_HEADS, FFN_DIM, name=f'encoder_{i}')
        x = enc(x)
    out = layers.Dense(FORECAST_HORIZON, name='projection')(x)
    out = tf.reshape(out, [-1, FORECAST_HORIZON])
    return keras.Model(inp, out)

model = build_freformer()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

## Train and evaluate

In [ ]:
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_train_flat = X_train.reshape(-1, 1)
X_test_flat = X_test.reshape(-1, 1)
y_train_flat = y_train.reshape(-1, 1)
X_train_s = scaler_x.fit_transform(X_train_flat).reshape(X_train.shape)
X_test_s = scaler_x.transform(X_test_flat).reshape(X_test.shape)
y_train_s = scaler_y.fit_transform(y_train_flat).reshape(y_train.shape)

BATCH = 128 if USE_GPU else 32
EPOCHS = 50
history = model.fit(X_train_s, y_train_s, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)

y_pred_s = model.predict(X_test_s, verbose=0)
y_pred_freformer = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).reshape(y_test.shape)
y_pred_freformer = np.clip(y_pred_freformer, 0, 100).astype(np.float32)

def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_f = calculate_mae(y_test, y_pred_freformer)
rmse_f = calculate_rmse(y_test, y_pred_freformer)
mase_f = calculate_mase(y_test, y_pred_freformer, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "FreFormer", "MAE": mae_f, "RMSE": rmse_f, "MASE": mase_f},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(results_df.to_string(index=False))
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: FreFormer (Frequency-Domain)', y=1.02, fontsize=12)
plt.show()

naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1: axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_freformer[i], '-', linewidth=1.6, label='FreFormer')
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted vs Actual')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_freformer.mean(axis=0), '-', linewidth=1.6, label='FreFormer (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

mae_per_hour = np.abs(y_test - y_pred_freformer).mean(axis=0)
mae_per_hour_n = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour, '-o', label='FreFormer', markersize=4)
ax.plot(hours, mae_per_hour_n, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residuals_f = (y_test - y_pred_freformer).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_f, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: FreFormer')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive')
plt.suptitle('Residual distribution', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): FreFormer = {residuals_f.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         FreFormer = {residuals_f.std():.4f}, Naive = {residuals_naive.std():.4f}')

best_row = results_df[results_df['Model'] == 'FreFormer'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"\nBest model: FreFormer (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}). Improvement over Naive: MAE {imp_mae:+.1f}%.")

### Key insights

- **Paper (CBD 2025):** FreFormer uses FFT + complex MLP (FFE) for intra-series frequency features and frequency-domain attention (FA) for inter-series dependencies; Add & Norm & FeedForward in each encoder layer.
- **Improvement over Naive:** Positive % means FreFormer beats the last-value baseline.
- **Univariate adaptation:** Here we use M=1 token (one series per sample); the same FFE/FA design applies; for multi-band joint modeling you could treat each band as a variable (M>1).
